# Charge Categorization QA

There's no ground-truth label for "correct category" here, so accuracy can't be checked against an answer key. Instead this notebook checks the classifier from `charge_categorization.ipynb` the way you'd audit a rule-based system:

1. **Coverage** — how many rows were classified by rule vs. TF-IDF fallback vs. not at all.
2. **Rule audit** — for every regex pattern in the taxonomy, which raw labels did it actually catch? This is where a too-generic pattern (e.g. matching `freight` inside something that isn't freight) would show up.
3. **Low-confidence fallback review** — the TF-IDF matches, sorted by similarity score so the weakest ones surface first.
4. **Consistency check** — does the same `Charge Type` ever get assigned to more than one Major Category (depending on its `Charge Description`)? That's a signal worth a manual look.
5. **Uncategorized bucket** — the full list of what's still in "Other / Uncategorized", not just the top 30.
6. **Random spot-check sample** — a reproducible random sample per major category to eyeball.

In [ ]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

mapping = pd.read_csv("outputs/charge_category_mapping.csv")
print(f"{len(mapping):,} unique (Charge Type, Charge Description) combinations, {mapping['row_count'].sum():,} rows total")
mapping.head()

## Taxonomy (must match `charge_categorization.ipynb` exactly — copied here so this notebook can re-derive *why* each combo was classified the way it was)

In [ ]:
TAXONOMY = {
    "Fuel Surcharge": {
        "Fuel Surcharge Correction": [r"correction.*fuel", r"fuel.*correction"],
        "Chargeback Fuel Surcharge": [r"chargeback.*fuel"],
        "International Fuel Surcharge": [r"worldease.*fuel", r"international.*fuel"],
        "Domestic Fuel Surcharge": [r"\bfuel surcharge\b", r"\bfuel\b", r"brandstof"],
    },
    "Accessorial / Delivery Surcharge": {
        "Delivery Area Surcharge (DAS)": [r"\bdas\b", r"delivery area surcharge"],
        "Residential Delivery/Surcharge": [r"residential"],
        "Signature Required": [r"signature"],
        "Additional Handling": [r"add'?l handling", r"additional handling", r"handling.*dimension"],
        "Saturday / After Hours": [r"saturday", r"after hours"],
        "Demand / Peak Surcharge": [r"demand surcharge", r"surge emergency", r"\bpeak\b"],
        "Wait Time / Mileage / Toll": [r"wait time", r"additional miles", r"\btoll\b"],
        "Address Correction": [r"address correction"],
        "Security Surcharge": [r"security surcharge"],
    },
    "Discounts & Credits": {
        "Earned Discount": [r"earned discount"],
        "Grace Discount": [r"grace discount"],
        "General Discount": [r"\bdiscount\b"],
        "Credit From Carrier": [r"credit from carrier"],
        "Chargeback / Reversal": [r"\bchargeback\b"],
        "Billing Adjustment / Correction": [r"billing adjustment", r"shipping charge correction", r"verzendcorrectiekosten"],
    },
    "Taxes & Customs": {
        "VAT": [r"\bvat\b", r"\bbtw\b"],
        "GST / HST": [r"\bgst\b", r"\bhst\b"],
        "Duty & Import Tax": [r"dut(y|ies)", r"import fee", r"import tax"],
        "Customs / Brokerage": [r"customs?", r"brokerage", r"export declaration", r"international processing", r"internationale verwerkingskosten", r"\beei\b"],
        "Sales / General Tax": [r"\btax(es)?\b"],
    },
    "Administrative & Service Fees": {
        "Disbursement Fee": [r"disbursement"],
        "Third Party Billing": [r"third party billing"],
        "Document Fee": [r"document fee", r"documents? preparation"],
        "Package Handling / Storage": [r"package handling", r"warehouse storage", r"\bstorage\b"],
        "Pickup Service": [r"pickup"],
    },
    "Base Freight / Transportation": {
        "Ground": [r"\bground\b", r"\bltl\b", r"truckload", r"road ?freight"],
        "Domestic Air (Next/2nd/3 Day)": [r"next day", r"2nd day", r"two day", r"3 day", r"third day", r"second day"],
        "International / Export / Import Freight": [r"worldwide express", r"ww express", r"\bexport\b", r"\bimport\b", r"world ?ease"],
        "Ocean Freight": [r"ocean"],
        "Line Haul": [r"line haul"],
        "Transportation / Base Charge": [r"transportation charge", r"\bbase\b", r"\bfreight\b", r"frt freight"],
    },
}
CATCH_ALL = ("Other / Uncategorized", "Unclassified")

SERVICE_TIER_OVERRIDES = [
    (r"\bground\b.*\bresidential\b", ("Base Freight / Transportation", "Ground")),
    (r"\b(next day|2nd day|second day|3 day|third day)\b.*\bresidential\b", ("Base Freight / Transportation", "Domestic Air (Next/2nd/3 Day)")),
]


def normalize(text):
    if pd.isna(text):
        return ""
    return str(text).lower().strip()

## 1. Coverage: rule vs. TF-IDF fallback vs. none

In [ ]:
coverage = mapping.groupby("Method").agg(combos=("Method", "size"), rows=("row_count", "sum"))
coverage["pct_of_rows"] = (coverage["rows"] / coverage["rows"].sum() * 100).round(1)
coverage.sort_values("rows", ascending=False)

## 2. Rule audit — which pattern caught what

Patterns match against `Charge Type` + `Charge Description` **combined into one string** (not Charge Type checked to completion first, then Charge Description as a fallback — that earlier version badly undercounted specific subcategories like "Ground" because a generic Charge Type like "Freight" matched the catch-all pattern before Charge Description was ever examined). For every combo classified by a rule, find the *exact* pattern that matched. Grouping by pattern shows you, per regex, how many rows it swept up and a few example raw labels — the fastest way to spot an over-broad pattern (e.g. a generic `\bbase\b` catching something that isn't a base freight charge).

In [ ]:
def rule_match_row(charge_type, charge_desc):
    t = (normalize(charge_type) + " " + normalize(charge_desc)).strip()
    if not t:
        return None
    for pat, result in SERVICE_TIER_OVERRIDES:
        if re.search(pat, t):
            return result + ("[override] " + pat,)
    for major, subcats in TAXONOMY.items():
        for sub, patterns in subcats.items():
            for pat in patterns:
                if re.search(pat, t):
                    return (major, sub, pat)
    return None


rule_rows = mapping[mapping["Method"] == "rule"].copy()
detail = rule_rows.apply(lambda r: rule_match_row(r["Charge Type"], r["Charge Description"]), axis=1)
rule_rows["Matched Pattern"] = [d[2] if d else None for d in detail]

pattern_summary = (
    rule_rows.groupby(["Major Category", "Subcategory", "Matched Pattern"])
    .agg(
        combos=("Matched Pattern", "size"),
        rows=("row_count", "sum"),
        examples=("Charge Type", lambda s: ", ".join(sorted(set(s.dropna().astype(str)))[:4])),
    )
    .sort_values("rows", ascending=False)
)
pattern_summary

### The riskiest patterns to check by hand

The generic, single-word patterns (`\bfreight\b`, `\bbase\b`, `\bdiscount\b`, `\btax(es)?\b`, etc.) are the ones most likely to catch something that doesn't belong. Pull every distinct raw label each one matched so you can eyeball them directly.

In [ ]:
GENERIC_PATTERNS = [r"\bfreight\b", r"\bbase\b", r"\bdiscount\b", r"\btax(es)?\b", r"\bstorage\b", r"\bpeak\b", r"\btoll\b"]

for pat in GENERIC_PATTERNS:
    sub = rule_rows[rule_rows["Matched Pattern"] == pat]
    if sub.empty:
        continue
    print(f"--- pattern `{pat}` -> {sub['row_count'].sum():,} rows, {len(sub)} combos ---")
    print(sub[["Charge Type", "Charge Description", "row_count"]].head(15).to_string(index=False))
    print()

## 3. TF-IDF fallback review

These combos didn't hit any keyword rule — they were assigned by nearest-neighbor similarity to a subcategory's reference keywords. **Do not trust the similarity score alone**: a systematic check found matches sharing just one generic word with a reference doc (e.g. "International Processing Fee" vs. the "International Fuel Surcharge" reference text) can score 0.4–0.7, comfortably above the 0.25 acceptance threshold, while being flatly wrong. So this section reviews fallback matches two ways: sorted by *similarity* (weakest first) and sorted by *row impact* (biggest dollar/row exposure first) — a wrong match sitting at 0.6 similarity but touching 5,000 rows is a bigger problem than a wrong match at 0.26 similarity touching 3 rows, and the similarity-only sort would never surface it.

In [ ]:
ref_docs, ref_labels = [], []
for major, subcats in TAXONOMY.items():
    for sub, patterns in subcats.items():
        doc = " ".join(p.replace(r"\b", "").replace(".*", " ").replace("'?", "").replace("?", "") for p in patterns)
        ref_docs.append(doc)
        ref_labels.append((major, sub))

vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
ref_vectors = vectorizer.fit_transform(ref_docs)


def best_similarity_row(charge_type, charge_desc):
    t = (normalize(charge_type) + " " + normalize(charge_desc)).strip()
    if not t:
        return None, None
    vec = vectorizer.transform([t])
    sims = cosine_similarity(vec, ref_vectors)[0]
    bi = sims.argmax()
    return ref_labels[bi], sims[bi]


fallback_rows = mapping[mapping["Method"] == "tfidf_fallback"].copy()

sims = []
for _, r in fallback_rows.iterrows():
    _, score = best_similarity_row(r["Charge Type"], r["Charge Description"])
    sims.append(score)

fallback_rows["similarity"] = sims

print(f"{len(fallback_rows):,} combos classified via TF-IDF fallback, {fallback_rows['row_count'].sum():,} rows")
print()
print("--- weakest matches (lowest similarity first) ---")
display(fallback_rows.sort_values("similarity")[["Charge Type", "Charge Description", "Major Category", "Subcategory", "similarity", "row_count"]].head(20))
print()
print("--- highest-impact matches (most rows first) -- review these even though confidence may look high ---")
display(fallback_rows.sort_values("row_count", ascending=False)[["Charge Type", "Charge Description", "Major Category", "Subcategory", "similarity", "row_count"]].head(20))

## 4. Consistency check — does the same Charge Type ever land in more than one Major Category?

Most Charge Types should map to exactly one category. If a Charge Type spans multiple categories depending on its Charge Description, that's worth a manual look — either it's a genuinely mixed bucket (carrier reuses one Charge Type label for several kinds of fees) or a pattern is over-firing on the Description field.

In [ ]:
type_consistency = (
    mapping.groupby("Charge Type")["Major Category"]
    .agg(lambda s: sorted(set(s)))
    .reset_index()
)
type_consistency["n_categories"] = type_consistency["Major Category"].apply(len)
inconsistent = type_consistency[type_consistency["n_categories"] > 1].sort_values("n_categories", ascending=False)
print(f"{len(inconsistent)} Charge Type values span more than one Major Category")
inconsistent

## 5. Full uncategorized list

Everything still in "Other / Uncategorized" — not caught by a rule or by the TF-IDF fallback threshold (0.25 similarity).

In [ ]:
uncategorized = mapping[mapping["Major Category"] == CATCH_ALL[0]].sort_values("row_count", ascending=False).reset_index(drop=True)
print(f"{len(uncategorized):,} uncategorized combinations, {uncategorized['row_count'].sum():,} rows, ${uncategorized['total_value'].sum():,.0f} total value")
uncategorized.to_csv("outputs/charge_uncategorized_full.csv", index=False)
print("Saved full list to outputs/charge_uncategorized_full.csv")
uncategorized.head(50)

## 6. Random spot-check sample

A reproducible random sample from each major category, for manually eyeballing whether the assignment makes sense.

In [ ]:
SEED = 42
sample = (
    mapping.groupby("Major Category", group_keys=False)
    .apply(lambda g: g.sample(n=min(5, len(g)), random_state=SEED))
    .sort_values(["Major Category", "row_count"], ascending=[True, False])
)
sample[["Major Category", "Subcategory", "Charge Type", "Charge Description", "Method", "row_count"]]